# Predicting Diabetes Disease Progression with Linear Regression

**Problem statement:** Given 10 baseline patient measurements (age, sex, BMI, blood pressure and six blood serum measures), predict a quantitative measure of disease progression one year after baseline. This is a **regression** task because the target is continuous.

**Dataset:** Scikit-learn's built-in Diabetes dataset (442 patients, 10 features, no missing values). Features are already mean-centred and scaled.

**Model:** Linear Regression, compared against a mean-predicting baseline.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Load and inspect the data

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Data types:")
print(df.dtypes.value_counts())
df.head()

In [ ]:
df.describe().T

## 3. Separate features (X) and target (y)

In [ ]:
X = df.drop(columns="target")   # 10 numeric features
y = df["target"]                # disease progression score

print("X shape:", X.shape, "| y shape:", y.shape)

## 4. Train/test split (80/20, fixed random state)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Train the baseline and Linear Regression models

In [ ]:
# Reference point: always predicts the training mean
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

# Chosen model
model = LinearRegression().fit(X_train, y_train)

coefs = pd.Series(model.coef_, index=X.columns).sort_values()
print("Intercept:", round(model.intercept_, 2))
coefs

## 6. Evaluate on the test set

In [ ]:
def evaluate(name, m):
    pred = m.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    return {"Model": name, "MSE": mse, "RMSE": np.sqrt(mse),
            "MAE": mean_absolute_error(y_test, pred), "R2": r2_score(y_test, pred)}

results = pd.DataFrame([evaluate("Mean baseline", baseline),
                        evaluate("Linear Regression", model)]).set_index("Model").round(3)
print("Train R2 (Linear Regression):", round(model.score(X_train, y_train), 3))
results

## 7. Visualise performance

In [ ]:
# Works locally (notebooks/ folder) and in Colab (any folder)
IMG_DIR = "../images" if os.path.basename(os.getcwd()) == "notebooks" else "images"

pred = model.predict(X_test)
resid = y_test - pred

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

ax[0].scatter(y_test, pred, alpha=0.7)
lo, hi = y.min(), y.max()
ax[0].plot([lo, hi], [lo, hi], "r--")
ax[0].set(xlabel="Actual", ylabel="Predicted", title="Actual vs Predicted")

ax[1].scatter(pred, resid, alpha=0.7)
ax[1].axhline(0, color="r", ls="--")
ax[1].set(xlabel="Predicted", ylabel="Residual", title="Residuals vs Predicted")

ax[2].hist(resid, bins=20, edgecolor="k")
ax[2].set(xlabel="Residual", ylabel="Count", title="Residual Distribution")

plt.tight_layout()
os.makedirs(IMG_DIR, exist_ok=True)
plt.savefig(f"{IMG_DIR}/evaluation_plots.png", dpi=150)   # saved for the repo
plt.show()

## 8. Interpretation

- Linear Regression cuts the test MSE roughly in half compared with the mean baseline (about 2900 vs 5362), so the features carry real signal.
- R² of about 0.45 means the model explains under half of the variance. It is useful as a first model but not highly accurate.
- Train R² (about 0.53) is close to test R², so there is no strong overfitting. The limit is the model's simplicity, not memorisation.
- s5, BMI, s2 and BP have large positive coefficients and s1 a large negative one. s1 and s2 are serum features that are likely correlated, so their individual coefficients should be read with caution.
- Next steps: try Ridge/Lasso, polynomial features, or cross-validation.